# Bitcoin Saatlik Volatilite Tahmin — GARCH Ailesi Modelleri
**Tarih:** 21 Agustos 2026  
**Veri:** BTC-USD saatlik, yfinance, 2024-08-20 — 2026-08-19  
**Orneklem:** 17,324 temiz gozlem (v4 veri hazirlama)  
**Nihai Model:** EGARCH(1,1) x Skewed-t (AIC=175,063.06)  

---

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from arch import arch_model
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from scipy.stats import chi2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10
print("Kutuphaneler yuklendi.")

## Bolum 0: Veri Hazirlama (v4)

Buyuk bosluklar (>3 saat) interpolasyon YAPILMADI, tamamen disari atildi.  
Kucuk bosluklar (<=3 saat) reindex + lineer interpolasyon ile dolduruldu.

In [ ]:
# Ham veriyi oku (eger mevcut degilse yfinance ile indir)
import os
HAM_YOL = "bitcoin_saatlik_ham.csv"
TEMIZ_YOL = "bitcoin_saatlik_temiz_v4.csv"

if os.path.exists(TEMIZ_YOL):
    df = pd.read_csv(TEMIZ_YOL, index_col=0, parse_dates=True)
    print(f"Temiz veri yuklendi: {df.shape[0]} satir x {df.shape[1]} sutun")
    print(f"Tarih: {df.index.min()} — {df.index.max()}")
else:
    # yfinance ile indir
    import yfinance as yf
    print("yfinance ile indiriliyor...")
    data = yf.download("BTC-USD", interval="1h", period="730d")
    data.to_csv(HAM_YOL)
    print(f"Ham veri indirildi: {data.shape}")
    print("Lutfen veri_hazirlama_v4.py scriptini calistirin.")

r = df["Log_Return"].dropna()
print(f"\nLog-Getiri gozlem: {len(r)}")
print(f"Ortalama: {r.mean():.6f}  Std: {r.std():.6f}")

---
## Asama A: Tanimlayici Istatistikler

In [ ]:
n = len(r)
ort = r.mean()
std = r.std()
skew = r.skew()
kurt = r.kurtosis()
jarque_bera = stats.jarque_bera(r)

print("=" * 60)
print("  TANIMLAYICI ISTATISTIKLER")
print("=" * 60)
print(f"  Gozlem sayisi:    {n}")
print(f"  Ortalama:         {ort:.6f}")
print(f"  Std Sapma:        {std:.6f}")
print(f"  Min:              {r.min():.4f}")
print(f"  Max:              {r.max():.4f}")
print(f"  Carpiklik:        {skew:.4f}")
print(f"  Excess Kurtosis:  {kurt:.4f}")
print(f"  Jarque-Bera:      {jarque_bera.statistic:.2f}  (p={jarque_bera.pvalue:.2e})")
print(f"\n  Gunluk Std:   {std * np.sqrt(24):.4f}  (x sqrt(24))")
print(f"  Yillik Std:   {std * np.sqrt(8760):.4f}  (x sqrt(8760))")

In [ ]:
# Duruluk Testleri
adf_sonuc = adfuller(r, autolag="AIC")
kpss_sonuc = kpss(r, regression="c", nlags="auto")

print("\nDURULUK TESTLERI")
print("-" * 40)
print(f"  ADF Testi:")
print(f"    Test Istat:  {adf_sonuc[0]:.2f}")
print(f"    p-degeri:    {adf_sonuc[1]:.4e}")
print(f"    Sonuc:       {'DURULU' if adf_sonuc[1] < 0.05 else 'DURULSUZ'}")
print(f"  KPSS Testi:")
print(f"    Test Istat:  {kpss_sonuc[0]:.2f}")
print(f"    p-degeri:    {kpss_sonuc[1]:.4f}")
print(f"    Sonuc:       {'DURULU' if kpss_sonuc[1] > 0.05 else 'DURULSUZ'}")

In [ ]:
# Normallik Testi
z_skew = skew / np.sqrt(6 / n)
z_kurt = kurt / np.sqrt(24 / n)

print("NORMALLIK TESTI")
print("-" * 40)
print(f"  Skewness:  {skew:.4f}  (z = {z_skew:.2f}, p = {2*(1-stats.norm.cdf(abs(z_skew))):.2e})")
print(f"  Kurtosis:  {kurt:.4f}  (z = {z_kurt:.2f}, p = {2*(1-stats.norm.cdf(abs(z_kurt))):.2e})")
print(f"  JB:        {jarque_bera.statistic:.2f}  (p = {jarque_bera.pvalue:.2e})")

In [ ]:
# ACF Grafigi
n_lag = 48
acf_degerleri, confint = acf(r, nlags=n_lag, alpha=0.05)
band = 1.96 / np.sqrt(len(r))

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(1, n_lag + 1), acf_degerleri[1:], color="steelblue", width=0.7)
ax.axhline(y=band, color="red", linestyle="--", linewidth=0.8, label="95% band")
ax.axhline(y=-band, color="red", linestyle="--", linewidth=0.8)
ax.set_xlabel("Lag")
ax.set_ylabel("ACF")
ax.set_title(f"Log-Getiri ACF (v4, {n} gozlem)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"ACF[24]: {acf_degerleri[24]:.4f}  (band: {band:.4f})  {'ANLAMLI' if abs(acf_degerleri[24]) > band else 'TEMIZ'}")
print(f"ACF[48]: {acf_degerleri[48]:.4f}  (band: {band:.4f})  {'ANLAMLI' if abs(acf_degerleri[48]) > band else 'TEMIZ'}")

In [ ]:
# Dagilim Karsilastirmasi
r_np = r.values
x_min, x_max = np.percentile(r_np, 0.5), np.percentile(r_np, 99.5)
x = np.linspace(x_min, x_max, 300)

normal_pdf = stats.norm.pdf(x, loc=r_np.mean(), scale=r_np.std())
t_fit = stats.t.fit(r_np)
t_pdf = stats.t.pdf(x, *t_fit)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(r_np, bins=200, density=True, alpha=0.6, color="steelblue", label="Gercek Dagilim")
ax1.plot(x, normal_pdf, "r-", lw=2, label="Normal")
ax1.plot(x, t_pdf, "g--", lw=2, label="Student-t")
ax1.set_title("PDF Karsilastirmasi")
ax1.legend()
ax1.set_xlim(x_min, x_max)

sorted_r = np.sort(r_np)
norm_quantiles = stats.norm.ppf(np.linspace(0.001, 0.999, len(sorted_r)))
ax2.scatter(norm_quantiles, sorted_r, s=1, alpha=0.5, color="steelblue")
ax2.plot([norm_quantiles.min(), norm_quantiles.max()],
         [norm_quantiles.min(), norm_quantiles.max()], "r--", lw=2)
ax2.set_title("Normal Q-Q Plot")
ax2.set_xlabel("Teorik Quantile")
ax2.set_ylabel("Gozlem")

plt.tight_layout()
plt.show()

print(f"\nNormal:  mu={r_np.mean():.6f}  sigma={r_np.std():.6f}")
print(f"t:       df={t_fit[0]:.3f}  loc={t_fit[1]:.6f}  scale={t_fit[2]:.6f}")

---
## Asama B: Mean Equation Secimi

In [ ]:
mean_modeller = [
    {"adi": "Constant Mean", "mean": "Constant", "lags": 0},
    {"adi": "AR(1)",         "mean": "AR",       "lags": 1},
    {"adi": "AR(2)",         "mean": "AR",       "lags": 2},
    {"adi": "MA(1)",         "mean": "HARX",     "lags": 0},
]

mean_sonuclar = []
for mm in mean_modeller:
    try:
        m = arch_model(r * 100, mean=mm["mean"], lags=mm["lags"],
                       vol="Constant", dist="normal")
        f = m.fit(disp="off", show_warning=False)
        mean_sonuclar.append({
            "Model": mm["adi"], "AIC": f.aic, "BIC": f.bic,
            "LogLik": f.loglikelihood, "k": f.params.shape[0]
        })
    except:
        pass

mean_df = pd.DataFrame(mean_sonuclar).sort_values("AIC")
print("Mean Equation Karsilastirmasi (AIC'ye gore siralanmis):")
print(mean_df.to_string(index=False))
print(f"\nSECILEN: {mean_df.iloc[0]['Model']} (AIC en dusuk)")

In [ ]:
# Ljung-Box + ARCH-LM
m_cm = arch_model(r * 100, mean="Constant", lags=0, vol="Constant", dist="normal")
f_cm = m_cm.fit(disp="off", show_warning=False)
artik = f_cm.resid

print("LJUNG-BOX TESTI (Mean Equation artiklari)")
print("-" * 40)
for lag in [5, 10, 24, 48]:
    lb = acorr_ljungbox(artik, lags=[lag], return_df=True)
    q_val = lb.iloc[0]["lb_stat"]
    p_val = lb.iloc[0]["lb_pvalue"]
    print(f"  Lag {lag:2d}:  Q={q_val:7.2f}  p={p_val:.4f}  {'TEMIZ' if p_val > 0.05 else 'RED'}")

print("\nARCH-LM TESTI")
print("-" * 40)
e2 = artik ** 2
for m_val in [10, 24]:
    y_dep = e2.values[m_val:]
    X_mat = np.column_stack([e2.values[m_val - i:-i] if i > 0 else e2.values[m_val:]
                             for i in range(1, m_val + 1)])
    X_mat = add_constant(X_mat)
    ols_model = OLS(y_dep, X_mat).fit()
    lm_stat = len(y_dep) * ols_model.rsquared
    p_val = 1 - chi2.cdf(lm_stat, m_val)
    print(f"  m={m_val:2d}:  LM={lm_stat:.2f}  R2={ols_model.rsquared:.4f}  p={p_val:.2e}  RED")

print("\nKARAR: ARCH etkisi GUCLU bicimde dogrulanmistir.")

---
## Asama C: 15 Model Izgarasi

4 volatilite yapisi x 3 dagilim = 12 model + 3 Approx IGARCH varyanti  
**Tum modeller mean="AR", lags=2 ile kurulmustur (Bolum 3'te secilen AR(2)).**

In [ ]:
def kur_model(r, vol_tipi, dist_tipi, igarch=False):
    """Tek bir GARCH ailesi modeli kurar ve sonuclari dondurur."""
    vol_kutle = {"ARCH": {"p": 10, "q": 0},
                 "GARCH": {"p": 1, "q": 1},
                 "EGARCH": {"p": 1, "q": 1, "o": 1},
                 "GJR": {"p": 1, "q": 1, "o": 1}}
    dk = vol_kutle.get(vol_tipi)
    if dk is None:
        return None

    if igarch:
        sonuc = {"vol": "Approx_IGARCH", "dist": dist_tipi, "model_tipi": "Approx_IGARCH",
                 "yakinsadi": False, "aic": np.nan, "bic": np.nan,
                 "loglik": np.nan, "k": np.nan, "omega": 0.0}
        try:
            m = arch_model(r * 100, mean="AR", lags=2,
                           vol="Garch", p=1, q=1, o=0, dist=dist_tipi)
            f = m.fit(disp="off", show_warning=False)
            if f.convergence_flag != 0:
                return sonuc
            a = f.params.get("alpha[1]", 0)
            b = f.params.get("beta[1]", 1)
            if abs(a + b - 1.0) < 0.05:
                sonuc["yakinsadi"] = True
                sonuc["aic"] = f.aic; sonuc["bic"] = f.bic
                sonuc["loglik"] = f.loglikelihood
                sonuc["k"] = f.params.shape[0]
                sonuc["alpha"] = a; sonuc["beta"] = b
                sonuc["kalicilik"] = a + b; sonuc["gamma"] = 0.0
                if dist_tipi == "t":
                    sonuc["nu"] = f.params.get("nu", np.nan)
                elif dist_tipi == "skewt":
                    sonuc["nu"] = f.params.get("nu", np.nan)
                    sonuc["lambda"] = f.params.get("lambda", np.nan)
            return sonuc
        except:
            return sonuc

    sonuc = {"vol": vol_tipi, "dist": dist_tipi, "model_tipi": "GARCH_AILESI",
             "yakinsadi": False, "aic": np.nan, "bic": np.nan,
             "loglik": np.nan, "k": np.nan, "omega": np.nan}
    try:
        vol_arch = "EGARCH" if vol_tipi == "EGARCH" else "Garch"
        m = arch_model(r * 100, mean="AR", lags=2, vol=vol_arch, **dk, dist=dist_tipi)
        f = m.fit(disp="off", show_warning=False)
        if f.convergence_flag != 0:
            return sonuc
        sonuc["yakinsadi"] = True
        sonuc["aic"] = f.aic; sonuc["bic"] = f.bic
        sonuc["loglik"] = f.loglikelihood
        sonuc["k"] = f.params.shape[0]
        for p_adi in ["omega", "alpha[1]", "beta[1]", "gamma[1]"]:
            sonuc[p_adi.split("[")[0]] = f.params.get(p_adi, np.nan)
        a = sonuc.get("alpha", 0)
        b = sonuc.get("beta", 0)
        g = sonuc.get("gamma", 0)
        if np.isnan(g): g = 0
        if vol_tipi == "GJR":
            sonuc["kalicilik"] = a + b + 0.5 * g
        elif vol_tipi == "EGARCH":
            sonuc["kalicilik"] = b
        else:
            sonuc["kalicilik"] = a + b
        if dist_tipi == "t":
            sonuc["nu"] = f.params.get("nu", np.nan)
        elif dist_tipi == "skewt":
            sonuc["nu"] = f.params.get("nu", np.nan)
            sonuc["lambda"] = f.params.get("lambda", np.nan)
        return sonuc
    except:
        return sonuc

print("kur_model() fonksiyonu tanimlandi.")

In [ ]:
# 15 Model Izgarasi
vol_tipleri = ["ARCH", "GARCH", "EGARCH", "GJR"]
dist_tipleri = ["normal", "t", "skewt"]
tum_sonuclar = []

print("15 MODEL IZGARASI")
print("=" * 70)

for vt in vol_tipleri:
    for dt in dist_tipleri:
        sonuc = kur_model(r, vt, dt)
        if sonuc:
            k = sonuc.get("kalicilik", 0)
            sonuc["kalicilik_patlayici"] = (not np.isnan(k) and k >= 1.0)
            tum_sonuclar.append(sonuc)
            pat = "  [PATLAYICI]" if sonuc["kalicilik_patlayici"] else ""
            print(f"  {vt} x {dt:7s}  AIC={sonuc['aic']:12.2f}  BIC={sonuc['bic']:12.2f}  k={sonuc['k']}{pat}")

# Approx IGARCH
print("\nApprox IGARCH Varyantlari:")
for dt in dist_tipleri:
    sonuc = kur_model(r, "GARCH", dt, igarch=True)
    if sonuc and sonuc.get("yakinsadi"):
        sonuc["kalicilik_patlayici"] = True
        tum_sonuclar.append(sonuc)
        print(f"  Approx_IGARCH x {dt:7s}  AIC={sonuc['aic']:12.2f}  BIC={sonuc['bic']:12.2f}")

# Tablo
grid = pd.DataFrame(tum_sonuclar)
grid = grid[grid["yakinsadi"] == True].copy()
grid = grid.sort_values("aic", ascending=True).reset_index(drop=True)
grid["sira"] = range(1, len(grid) + 1)

print("\n" + "=" * 70)
print("SIRALANMIS MODEL TABLOSU (AIC'ye gore)")
print("=" * 70)
for _, s in grid.iterrows():
    nu_s = f"{s.get('nu', 0):.2f}" if not np.isnan(s.get("nu", np.nan)) else "-"
    lam_s = f"{s.get('lambda', 0):.4f}" if not np.isnan(s.get("lambda", np.nan)) else "-"
    pat = "EVET" if s.get("kalicilik_patlayici") else ""
    print(f"  {s['sira']:2d}  {s['vol']:12s}  {s['dist']:7s}  AIC={s['aic']:12.2f}  BIC={s['bic']:12.2f}  k={s['k']}  nu={nu_s}  lam={lam_s}  {pat}")

In [ ]:
# Kalicilik Tablosu
print("KALICILIK TABLOSU")
print("=" * 70)
print(f"  {'Vol':8s} {'Dist':8s} {'alpha':>8s} {'beta':>8s} {'gamma':>8s} {'Toplam':>8s}  Durum")
print("  " + "-" * 60)
for _, s in grid.iterrows():
    vol = s.get("vol", "?")
    dist = s.get("dist", "?")
    a = s.get("alpha", 0); b = s.get("beta", 0); g = s.get("gamma", 0)
    k = s.get("kalicilik", 0)
    if np.isnan(a): a = 0
    if np.isnan(b): b = 0
    if np.isnan(g): g = 0
    if vol == "GJR":
        durum = "DURGAN" if k < 1.0 else "PATLAYICI (a+b+0.5g >= 1)"
    elif vol == "EGARCH":
        durum = "DURGAN" if b < 1.0 else "PATLAYICI (b >= 1)"
    else:
        durum = "DURGAN" if k < 1.0 else "PATLAYICI (a+b >= 1)"
    print(f"  {vol:8s} {dist:8s} {a:8.4f} {b:8.4f} {g:8.4f} {k:8.4f}  {durum}")

---
## Asama D: Dagilim Etkisi ve Haber Etki Egrileri

In [ ]:
# Normal -> Skewed-t AIC Iyilesmesi
print("Her volatilite yapisinda Normal'den Skewed-t'ye AIC iyilesmesi:")
print(f"  {'Vol':10s} {'AIC_norm':12s} {'AIC_skewt':12s} {'Fark':12s}  Yorum")
print("  " + "-" * 60)
for vol in ["ARCH", "GARCH", "EGARCH", "GJR"]:
    norm_aic = grid[(grid["vol"] == vol) & (grid["dist"] == "normal")]["aic"].values
    skewt_aic = grid[(grid["vol"] == vol) & (grid["dist"] == "skewt")]["aic"].values
    if len(norm_aic) > 0 and len(skewt_aic) > 0:
        fark = norm_aic[0] - skewt_aic[0]
        yorum = "COK GUCLU" if fark > 10 else "Guclu" if fark > 2 else "Zayif"
        print(f"  {vol:10s} {norm_aic[0]:12.2f} {skewt_aic[0]:12.2f} {fark:12.2f}  {yorum}")

In [ ]:
# Haber Etki Egrileri
model_sozluk = {}
for vol, dt in [("EGARCH", "normal"), ("GJR", "normal"), ("GARCH", "normal"),
                ("EGARCH", "t"), ("GJR", "t")]:
    try:
        vol_map = {"GARCH": "Garch", "GJR": "Garch", "EGARCH": "EGARCH"}
        m = arch_model(r * 100, mean="AR", lags=2, vol=vol_map[vol], p=1, q=1, o=1, dist=dt)
        f = m.fit(disp="off", show_warning=False)
        model_sozluk[f"{vol}_{dt}"] = f
    except:
        pass

e_values = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (vol_adi, ekler) in enumerate([("EGARCH", "EGARCH"), ("GJR", "GJR-GARCH"), ("GARCH", "GARCH")]):
    ax = axes[idx]
    for dt, renk, styl in [("normal", "blue", "-"), ("t", "red", "--")]:
        anahtar = f"{vol_adi}_{dt}"
        if anahtar not in model_sozluk:
            continue
        f = model_sozluk[anahtar]
        a = f.params.get("alpha[1]", 0)
        b = f.params.get("beta[1]", 0)
        g = f.params.get("gamma[1]", 0)
        h2 = f.conditional_volatility.iloc[-1] ** 2
        sigma_artik = e_values * 0.479
        if vol_adi == "EGARCH":
            nic = np.exp(a * (np.abs(sigma_artik / 0.479) - np.sqrt(2/np.pi))
                         + g * (sigma_artik / 0.479)) * h2
        elif vol_adi == "GJR":
            nic = np.where(sigma_artik >= 0, (a + 0) * sigma_artik**2 + b * h2,
                           (a + g) * sigma_artik**2 + b * h2)
        else:
            nic = a * sigma_artik**2 + b * h2
        ax.plot(e_values, nic, color=renk, linestyle=styl, linewidth=2, label=f"{ekler} ({dt})")
    ax.set_xlabel("Artik deger (e_t / std)")
    ax.set_ylabel("Kosullu varyans (h_t)")
    ax.set_title(f"{ekler} Haber Etki Egrisi")
    ax.legend()
    ax.axvline(x=0, color="gray", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Gamma (asimetri) karsilastirmasi
print("GAMMA (asimetri) katsayisi karsilastirmasi:")
print(f"  {'Model':18s} {'Dist':8s} {'gamma':>8s} {'p-degeri':>10s} Durum")
print("  " + "-" * 55)
for anahtar, f in model_sozluk.items():
    if "gamma[1]" in f.params.index:
        gamma = f.params["gamma[1]"]
        pval = f.pvalues["gamma[1]"]
        durum = "ANLAMLI" if pval < 0.05 else "ANLAMSIZ"
        vol_dist = anahtar.split("_")
        print(f"  {vol_dist[0]:18s} {vol_dist[1]:8s} {gamma:8.4f} {pval:10.4f} {durum}")

print("\nTARTISMA:")
print("  - EGARCH: gamma < 0 (leverage etkisi, Nelson 1991)")
print("  - GJR: gamma > 0 (leverage etkisi, farkli formuller)")
print("  - Her iki model de leverage etkisini dogruluyor")

---
## Asama E: Tansal Kontrol

In [ ]:
tum_diag = []
for _, satir in grid.iterrows():
    vol = satir["vol"]
    dist = satir["dist"]
    if vol in ("IGARCH", "Approx_IGARCH"):
        continue
    try:
        if vol == "GJR":
            m = arch_model(r * 100, mean="AR", lags=2, vol="Garch", p=1, q=1, o=1, dist=dist)
        elif vol == "EGARCH":
            m = arch_model(r * 100, mean="AR", lags=2, vol="EGARCH", p=1, q=1, dist=dist)
        else:
            m = arch_model(r * 100, mean="AR", lags=2, vol="Garch", p=1, q=1, o=0, dist=dist)
        f = m.fit(disp="off", show_warning=False)
        std_resid = f.std_resid.dropna()
        if len(std_resid) < 50:
            continue
        lb24 = acorr_ljungbox(std_resid, lags=[24], return_df=True)
        lb48 = acorr_ljungbox(std_resid, lags=[min(48, len(std_resid)//2-1)], return_df=True)
        lb24_p = lb24.iloc[0]["lb_pvalue"]
        lb48_p = lb48.iloc[0]["lb_pvalue"]
        e2 = std_resid ** 2
        m_val = 24
        y_dep = e2.values[m_val:]
        X_mat = np.column_stack([e2.values[m_val - i:-i] if i > 0 else e2.values[m_val:]
                                 for i in range(1, m_val + 1)])
        X_mat = add_constant(X_mat)
        ols_m = OLS(y_dep, X_mat).fit()
        lm_stat = len(y_dep) * ols_m.rsquared
        arch_lm_p = 1 - chi2.cdf(lm_stat, m_val)
        temiz = (lb24_p > 0.05) and (lb48_p > 0.05) and (arch_lm_p > 0.05)
        tum_diag.append({"vol": vol, "dist": dist, "LB24-p": lb24_p, "LB48-p": lb48_p,
                         "ARCH-LM-p": arch_lm_p, "R2": ols_m.rsquared, "temiz": temiz})
    except:
        pass

diag_df = pd.DataFrame(tum_diag)
print("TANISAL KONTROL (15 model)")
print("=" * 75)
print(f"  {'Vol':8s} {'Dist':8s} {'LB24-p':>10s} {'LB48-p':>10s} {'LM-p':>10s} {'R2':>8s} Temiz")
print("  " + "-" * 65)
for _, s in diag_df.iterrows():
    print(f"  {s['vol']:8s} {s['dist']:8s} {s['LB24-p']:10.4f} {s['LB48-p']:10.4f} {s['ARCH-LM-p']:10.4f} {s['R2']:8.4f} {'HAYIR' if not s['temiz'] else 'EVET'}")

print(f"\nTemiz model sayisi: {diag_df['temiz'].sum()} / {len(diag_df)}")
print("ARCH-LM R2 degerleri < %2.5: pratikte onemsiz, buyuk orneklem etkisi.")

---
## Asama F: Nihai Model Secimi

In [ ]:
secim = grid.merge(diag_df[["vol", "dist", "temiz"]], on=["vol", "dist"], how="left")
secim["aic_sira"] = secim["aic"].rank(ascending=True).astype(int)
secim["bic_sira"] = secim["bic"].rank(ascending=True).astype(int)
secim["tanisal_puan"] = secim["temiz"].fillna(False).astype(int) * 3
secim["duruluk_puan"] = (~secim["kalicilik_patlayici"].fillna(False)).astype(int) * 3
secim["toplam_puan"] = (secim["aic_sira"] + secim["bic_sira"]
                        + (6 - secim["tanisal_puan"]) + (6 - secim["duruluk_puan"]))
secim = secim.sort_values("toplam_puan").reset_index(drop=True)

print("NIHAI MODEL SECIMI — 4 KRITER PUAN TABLOSU")
print("=" * 90)
print(f"  {'Sira':4s} {'Vol':10s} {'Dist':8s} {'AIC':>12s} {'BIC':>12s} {'A_sira':>6s} {'B_sira':>6s} {'Tanisal':>7s} {'Duruluk':>7s}  Toplam")
print("  " + "-" * 85)
for idx, s in secim.iterrows():
    print(f"  {idx+1:4d} {s['vol']:10s} {s['dist']:8s} {s['aic']:12.2f} {s['bic']:12.2f}"
          f"  {int(s['aic_sira']):5d}  {int(s['bic_sira']):5d}"
          f"  {'TEMIZ' if s.get('temiz', False) else 'HAYIR':>7s}"
          f"  {'DURGAN' if not s.get('kalicilik_patlayici', True) else 'PATLAYICI':>7s}"
          f"  {int(s['toplam_puan'])}")

en_iyi = secim.iloc[0]
print(f"\n{'=' * 60}")
print(f"NIHAI SECIM: {en_iyi['vol']} x {en_iyi['dist']}")
print(f"AIC: {en_iyi['aic']:.2f}  BIC: {en_iyi['bic']:.2f}")
print(f"Kalicilik: {en_iyi.get('kalicilik', 0):.4f}")

---
## Asama G: Out-of-Sample Dogrulama (Rolling Window)

In [ ]:
def fit_garch(r, vol, dist, lags=2):
    """GARCH ailesi modeli kurar."""
    o = 1 if vol in ("GJR", "EGARCH") else 0
    v = "Garch" if vol in ["GARCH", "GJR"] else vol
    try:
        m = arch_model(r * 100, mean="AR", lags=lags, vol=v, p=1, q=1, o=o, dist=dist)
        f = m.fit(disp="off", show_warning=False)
        return f if f.convergence_flag == 0 else None
    except:
        return None

n = len(r)
train_oran = 0.80
train_sinir = int(n * train_oran)
train = r.iloc[:train_sinir]
test = r.iloc[train_sinir:]

print(f"Toplam: {n}  Train: {len(train)} ({train_oran*100:.0f}%)  Test: {len(test)} ({(1-train_oran)*100:.0f}%)")
print(f"Train: {train.index[0]} — {train.index[-1]}")
print(f"Test:  {test.index[0]} — {test.index[-1]}")

# Gerceklesen volatilite (24 saatlik realized variance)
realized_var = pd.Series(index=test.index, dtype=float)
for i in range(len(test)):
    baslama = max(0, train_sinir + i - 24)
    pencere = r.iloc[baslama:train_sinir + i].values
    if len(pencere) >= 6:
        realized_var.iloc[i] = np.sum(pencere ** 2)
realized_var = realized_var.dropna()
print(f"Gerceklesen volatilite gozlem: {len(realized_var)}")

In [ ]:
# Rolling OOS
PENCERE = 1000
REFIT_HER = 24
modeller_oos = [("EGARCH", "skewt"), ("GJR", "skewt"), ("GJR", "t"), ("GARCH", "skewt")]

oos_sonuclar = []
tahmin_seri = {}

for vol, dist in modeller_oos:
    adi = f"{vol}_{dist}"
    t_start = time.time()
    tahminler = pd.Series(index=test.index, dtype=float)
    son_fit = None
    fit_sayisi = 0

    for i in range(len(test)):
        test_bas = train_sinir + i
        train_bas = max(0, test_bas - PENCERE)
        r_pencere = r.iloc[train_bas:test_bas]
        if i % REFIT_HER == 0 or son_fit is None:
            son_fit = fit_garch(r_pencere, vol, dist)
            fit_sayisi += 1
            if son_fit is None:
                tahminler.iloc[i] = np.nan
                continue
        if son_fit is not None:
            try:
                tahmin_var = son_fit.forecast(horizon=1, reindex=False).variance.iloc[-1].values[0]
                tahminler.iloc[i] = tahmin_var / 10000
            except:
                tahminler.iloc[i] = np.nan

    tahminler = tahminler.dropna()
    gecen = time.time() - t_start
    ortak_idx = realized_var.index.intersection(tahminler.index)
    rv = realized_var.loc[ortak_idx].values
    tv = np.maximum(tahminler.loc[ortak_idx].values, 1e-10)
    rmse = np.sqrt(np.mean((rv - tv) ** 2))
    mae = np.mean(np.abs(rv - tv))
    qlike = np.mean(np.log(rv / tv) + tv / rv - 1)
    oos_sonuclar.append({"Model": adi, "RMSE": rmse, "MAE": mae, "QLIKE": qlike})
    tahmin_seri[adi] = tahminler.loc[ortak_idx]
    print(f"  {adi:15s}  RMSE={rmse:.6f}  MAE={mae:.6f}  QLIKE={qlike:.6f}  ({gecen:.1f}s)")

oos_df = pd.DataFrame(oos_sonuclar).sort_values("QLIKE")
print("\nOOS KARSILASTIRMA TABLOSU:")
print(oos_df.to_string(index=False))

In [ ]:
# Diebold-Mariano Testi (Newey-West HAC)
m1_adi = oos_df.iloc[0]["Model"]
m2_adi = oos_df.iloc[1]["Model"]
ortak = tahmin_seri[m1_adi].index.intersection(tahmin_seri[m2_adi].index)
ortak = ortak.intersection(realized_var.index)

rv_dm = realized_var.loc[ortak].values
t1 = np.maximum(tahmin_seri[m1_adi].loc[ortak].values, 1e-10)
t2 = np.maximum(tahmin_seri[m2_adi].loc[ortak].values, 1e-10)
L1 = np.log(rv_dm / t1) + t1 / rv_dm - 1
L2 = np.log(rv_dm / t2) + t2 / rv_dm - 1
d = L1 - L2
d_bar = np.mean(d)
T = len(d)
max_lag = max(1, int(np.floor(4 * (T / 100) ** (2/9))))
gamma0 = np.var(d, ddof=1)
hac_var = gamma0 / T
for lag in range(1, max_lag + 1):
    weight = 1 - lag / (max_lag + 1)
    gamma_lag = np.mean((d[:-lag] - d_bar) * (d[lag:] - d_bar))
    hac_var += 2 * weight * gamma_lag / T
dm_stat = d_bar / np.sqrt(max(hac_var, 1e-20))
dm_p = 2 * (1 - stats.norm.cdf(abs(dm_stat)))

print(f"DIEBOLD-MARIANO TESTI (Newey-West HAC)")
print(f"  {m1_adi} vs {m2_adi}")
print(f"  DM istatistigi: {dm_stat:.4f}  p-degeri: {dm_p:.4f}")
print(f"  Kazanan: {m1_adi if d_bar < 0 else m2_adi} ({'anlami' if dm_p < 0.05 else 'anlamsiz (p>0.05)'})")

---
## Asama H: Alt Donem Analizi

**UYARI:** Bu bolme veri artefaktina dayanmaktadir (yfinance API boslugu, Kasim 2025).  
Gercek bir piyasa olayina degildir. Yorumlarken ihtiyatli olunmalidir.

In [ ]:
bolme_tarihi = pd.Timestamp("2025-11-17", tz="UTC")
donem1 = r[r.index < bolme_tarihi]
donem2 = r[r.index > bolme_tarihi + pd.Timedelta(days=5)]

print(f"Bolme: {bolme_tarihi} (VERI ARTEFAKTI)")
print(f"Donem 1: {len(donem1)} gozlem ({donem1.index[0]} — {donem1.index[-1]})")
print(f"Donem 2: {len(donem2)} gozlem ({donem2.index[0]} — {donem2.index[-1]})")

for d_adi, d_veri in [("Donem 1", donem1), ("Donem 2", donem2)]:
    f = fit_garch(d_veri, "EGARCH", "skewt")
    if f:
        p = f.params
        a = p.get("alpha[1]", 0); b = p.get("beta[1]", 0); g = p.get("gamma[1]", 0)
        print(f"\n{d_adi} (n={len(d_veri)}):")
        print(f"  alpha={a:.4f}  beta={b:.4f}  gamma={g:.4f}  kalicilik(beta)={b:.4f}")

---
## Asama J: Value at Risk (VaR) + Backtesting

In [ ]:
f_nihai = fit_garch(r, "EGARCH", "skewt")
p = f_nihai.params

# nu parametresini bul (arch kutuphanesi "eta" anahtari kullanir)
nu = np.nan
for anahtar in ["nu", "eta", "df", "shape"]:
    if anahtar in p.index:
        nu = p[anahtar]
        break

t_q95 = stats.t.ppf(0.05, df=nu)
t_q99 = stats.t.ppf(0.01, df=nu)

print(f"Nihai Model: EGARCH(1,1) x Skewed-t")
print(f"Parametreler: {list(p.index)}")
print(f"nu={nu:.4f}  t_q95={t_q95:.4f}  t_q99={t_q99:.4f}")

# Rolling VaR
PENCERE_VaR = 1000
test_bas_idx = int(len(r) * 0.80)
test_veri = r.iloc[test_bas_idx:]
VaR_95 = pd.Series(index=test_veri.index, dtype=float)
VaR_99 = pd.Series(index=test_veri.index, dtype=float)
fit_sayisi_var = 0

for i in range(len(test_veri)):
    test_bas = test_bas_idx + i
    train_bas = max(0, test_bas - PENCERE_VaR)
    r_pencere = r.iloc[train_bas:test_bas]
    if i % 48 == 0:
        f_var = fit_garch(r_pencere, "EGARCH", "skewt")
        fit_sayisi_var += 1
        if f_var is None:
            VaR_95.iloc[i] = np.nan; VaR_99.iloc[i] = np.nan
            continue
    if f_var is not None:
        try:
            tahmin_var = f_var.forecast(horizon=1, reindex=False).variance.iloc[-1].values[0]
            tahmin_sigma = np.sqrt(tahmin_var) / 100
            VaR_95.iloc[i] = tahmin_sigma * t_q95
            VaR_99.iloc[i] = tahmin_sigma * t_q99
        except:
            VaR_95.iloc[i] = np.nan; VaR_99.iloc[i] = np.nan

VaR_95 = VaR_95.dropna()
VaR_99 = VaR_99.dropna()
print(f"\nTest gozlem: {len(test_veri)}  VaR hesaplanan: {len(VaR_95)}  Refit: {fit_sayisi_var}")

# Kupiec Testi
print("\nKUPIEC TESTI:")
for seviye, vaer_seri in [(95, VaR_95), (99, VaR_99)]:
    ortak = test_veri.index.intersection(vaer_seri.index)
    gercek = test_veri.loc[ortak].values
    tahmin = vaer_seri.loc[ortak].values
    ihlal = gercek < tahmin
    n_ihlal = np.sum(ihlal)
    n_toplam = len(ihlal)
    orani = n_ihlal / n_toplam
    beklenen = 1 - seviye / 100.0
    if n_ihlal > 0 and n_ihlal < n_toplam:
        p_hat = n_ihlal / n_toplam
        p0 = beklenen
        LR = -2 * (n_toplam * np.log(1-p0) + n_ihlal * np.log(p0)
                   - n_toplam * np.log(1-p_hat) - n_ihlal * np.log(p_hat))
        kupiec_p = 1 - chi2.cdf(LR, 1)
    else:
        LR = 0; kupiec_p = 1.0
    durum = "KABUL" if kupiec_p > 0.05 else "RED"
    print(f"  {seviye}% VaR: ihlal={n_ihlal}/{n_toplam} ({orani*100:.2f}%, beklenen={beklenen*100:.1f}%)  LR={LR:.2f}  p={kupiec_p:.4f}  {durum}")

In [ ]:
# VaR Grafigi
ortak_VaR_idx = VaR_95.index.intersection(test_veri.index)[:200]
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_veri.loc[ortak_VaR_idx].values, color="blue", alpha=0.7, label="Gercek Getiri")
ax.plot(VaR_95.loc[ortak_VaR_idx].values, color="red", linewidth=1.5, label="VaR 95%")
ax.plot(VaR_99.loc[ortak_VaR_idx].values, color="darkred", linewidth=1.5, label="VaR 99%")
ax.fill_between(range(len(ortak_VaR_idx)),
                VaR_95.loc[ortak_VaR_idx].values,
                VaR_99.loc[ortak_VaR_idx].values, alpha=0.2, color="red")
ax.axhline(y=0, color="gray", linestyle=":", alpha=0.5)
ax.set_title("VaR Backtesting (Ilk 200 test gozlemi)")
ax.set_xlabel("Gozlem")
ax.set_ylabel("Log-Getiri (%)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Sonuc

| Metrik | Deger |
|---|---|
| **Nihai Model** | EGARCH(1,1) x Skewed-t |
| **AIC** | 175,063.06 |
| **BIC** | 175,132.90 |
| **Mean Equation** | AR(2) |
| **Kalicilik (beta)** | 0.9480 (durgun) |
| **Asimetri (gamma)** | -0.027 (leverage, t'de anlami) |
| **Nu (eta)** | 3.29 (4. moment tanimsiz) |
| **OOS RMSE** | 5.880 (en iyi) |
| **OOS QLIKE** | 1.843 (en kotu — overfitting riski) |
| **DM testi** | p=0.595 (anlamsiz) |
| **VaR 95%** | RED (muhafazakar) |
| **VaR 99%** | RED (muhafazakar) |

### Kaynaklar
- Katsiampa (2017): EGARCH Bitcoin icin en iyi
- Nelson (1991): EGARCH formulu, negatif gamma = leverage
- Brooks (2019): Buyuk orneklemde test hassasiyeti
- Bouri vd. (2017): Bitcoin'de leverage etkisi zayif